## Q4: Put Sum/ Max/ Min & Avg of elements of vector hA in hSum/ hMax/ hMin & hAvg 

In [1]:
%%writefile q4.cu

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N          60
#define LOW        10
#define HIGH       99
#define BLOCK_SIZE  8

// ── CUDA Kernels
__global__ void sumKernel(int *dA, int *dSum, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        atomicAdd(dSum, dA[i]);         /* atomic to avoid race condition */
}

__global__ void maxKernel(int *dA, int *dMax, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        atomicMax(dMax, dA[i]);         /* atomic max */
}

__global__ void minKernel(int *dA, int *dMin, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        atomicMin(dMin, dA[i]);         /* atomic min */
}

// ── Host Functions 
__host__ int vecSum(int *hA, int n)
{
    int *dA, *dSum;
    int  hSum = 0;
    int  size = n * sizeof(int);

    cudaMalloc((void**)&dA,   size);
    cudaMalloc((void**)&dSum, sizeof(int));

    cudaMemcpy(dA,   hA,   size,         cudaMemcpyHostToDevice);
    cudaMemcpy(dSum, &hSum, sizeof(int), cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, 1, 1);
    dim3 DimGrid((int)ceil((float)n / BLOCK_SIZE), 1, 1);

    sumKernel<<<DimGrid, DimBlock>>>(dA, dSum, n);

    cudaMemcpy(&hSum, dSum, sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(dA); cudaFree(dSum);
    return hSum;
}

__host__ int vecMax(int *hA, int n)
{
    int *dA, *dMax;
    int  hMax = hA[0];
    int  size = n * sizeof(int);

    cudaMalloc((void**)&dA,   size);
    cudaMalloc((void**)&dMax, sizeof(int));

    cudaMemcpy(dA,   hA,   size,          cudaMemcpyHostToDevice);
    cudaMemcpy(dMax, &hMax, sizeof(int),  cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, 1, 1);
    dim3 DimGrid((int)ceil((float)n / BLOCK_SIZE), 1, 1);

    maxKernel<<<DimGrid, DimBlock>>>(dA, dMax, n);

    cudaMemcpy(&hMax, dMax, sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(dA); cudaFree(dMax);
    return hMax;
}

__host__ int vecMin(int *hA, int n)
{
    int *dA, *dMin;
    int  hMin = hA[0];
    int  size = n * sizeof(int);

    cudaMalloc((void**)&dA,   size);
    cudaMalloc((void**)&dMin, sizeof(int));

    cudaMemcpy(dA,   hA,   size,          cudaMemcpyHostToDevice);
    cudaMemcpy(dMin, &hMin, sizeof(int),  cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, 1, 1);
    dim3 DimGrid((int)ceil((float)n / BLOCK_SIZE), 1, 1);

    minKernel<<<DimGrid, DimBlock>>>(dA, dMin, n);

    cudaMemcpy(&hMin, dMin, sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(dA); cudaFree(dMin);
    return hMin;
}

// ── Main 
int main()
{
    int   *hA;
    int    hSum, hMax, hMin;
    float  hAvg;

    hA = (int*) malloc(N * sizeof(int));

    srand(time(NULL));
    for(int i=0; i<N; i++)
    {
        hA[i] = rand()%90+10;
    }

    printf("hA: "); for(int i=0;i<N;i++) printf("%4d",hA[i]); printf("\n");

    hSum = vecSum(hA, N);
    hMax = vecMax(hA, N);
    hMin = vecMin(hA, N);
    hAvg = (float)hSum / N;         /* Avg computed from Sum */

    printf("\nSum (hSum) = %d\n",   hSum);
    printf("Max (hMax) = %d\n",     hMax);
    printf("Min (hMin) = %d\n",     hMin);
    printf("Avg (hAvg) = %.2f\n",   hAvg);

    free(hA);
    return 0;
}

Writing q4.cu


## Compile and Run:

In [2]:
!nvcc -arch=sm_75 q4.cu -o q4

!./q4

hA:   26  96  52  53  11  58  72  27  75  37  92  85  75  99  86  14  24  50  86  70  26  62  57  49  71  78  83  16  84  56  57  63  14  99  16  15  20  78  84  47  67  39  33  95  38  71  99  14  22  85  74  90  47  83  91  70  62  75  38  46

Sum (hSum) = 3502
Max (hMax) = 99
Min (hMin) = 11
Avg (hAvg) = 58.37
